In [33]:
import os
import json
from transformers import DataCollatorForLanguageModeling, TrainingArguments, Trainer, AutoModelForMaskedLM, AutoTokenizer
import numpy as np
from datasets import Dataset, load_dataset, load_from_disk

In [ ]:
# os.environ['PYTORCH_MPS_HIGH_WATERMARK_RATIO'] = '0.0'

In [2]:
tokenizer = AutoTokenizer.from_pretrained("distilbert/distilroberta-base")
model = AutoModelForMaskedLM.from_pretrained("distilbert/distilroberta-base")

Some weights of the model checkpoint at distilbert/distilroberta-base were not used when initializing RobertaForMaskedLM: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [3]:
all_summaries = []
second_split = ['Title:', 'Summary:']
for fname in os.listdir('all_jsons'):
    if not fname.endswith('.json'):
        continue
    with open('all_jsons/' + fname, 'r') as f:
        data = json.load(f)
    for item in data:
        text = data[item]
        for i, keyword in enumerate(['Published:', 'Page:']):
            if keyword in text:
                break
        for txt in text.split(keyword):
            txt = txt.strip()
            if txt == '':
                continue
            try:
                summary = txt.split(second_split[i])[1].strip()
                if summary == '':
                    continue
            except:
                continue
            all_summaries.append(summary)
all_summaries = np.unique(all_summaries)

In [4]:
max_input_length = 512

In [5]:
def preprocess_function(examples):
    return tokenizer(examples['text'], max_length=max_input_length, truncation=True)

In [6]:
ds = Dataset.from_list([{'text': txt} for txt in all_summaries])
ds = ds.map(preprocess_function)
ds = ds.train_test_split(test_size=0.2)
ds

Map:   0%|          | 0/5239 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'input_ids', 'attention_mask'],
        num_rows: 4191
    })
    test: Dataset({
        features: ['text', 'input_ids', 'attention_mask'],
        num_rows: 1048
    })
})

In [14]:
ds['train'].save_to_disk("train.hf")
ds['test'].save_to_disk("test.hf")

Saving the dataset (0/1 shards):   0%|          | 0/4191 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1048 [00:00<?, ? examples/s]

In [20]:
from datasets import load_from_disk
train_ds = load_from_disk('train.hf')
test_ds = load_from_disk('test.hf')
print(train_ds[0]['text'])

Intrinsic aeroacoustic instabilities in the crosstalk apertures of can-annular combustors Authors: Audrey Blondé, Khushboo Pandey, Bruno Schuermans, Nicolas Noiray Summary: This paper presents an experimental and numerical study of aeroacoustic instabilities at the interface between neighbouring combustion chambers in modern heavy-duty gas turbines. A simplified laboratory-scale geometry of the gap separating the outlet of these chambers, just


In [21]:
!open .

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [7]:
tokenizer.pad_token = tokenizer.eos_token
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm_probability=0.15)

In [ ]:
training_args = TrainingArguments(
    output_dir="mlm_industrial_emb",
    eval_strategy="epoch",
    learning_rate=2e-5,
    num_train_epochs=3,
    weight_decay=0.01,
    per_device_train_batch_size=1,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=ds["train"],
    eval_dataset=ds["test"],
    data_collator=data_collator,
    tokenizer=tokenizer,
)

trainer.train()